In [0]:
# Databricks notebook source
import os
# COMMAND ----------
CATALOG = "data_warehouse_factory"
SCHEMA = "bronze"

# Lista tabel i odpowiadających im katalogów w Volumes
TABLES = [
    ("bronze_employees", "employees_raw"),
    ("bronze_employee_assignments", "employee_assignments_raw"),
    ("bronze_production_structure", "lines_cells_raw"),
    ("bronze_production_plan", "production_plan_raw"),
    ("bronze_events", "capacity_planner_events_raw"),
]

for table_name, volume_folder in TABLES:
    target_table = f"{CATALOG}.{SCHEMA}.{table_name}"
    volume_path = f"/Volumes/{CATALOG}/{SCHEMA}/{volume_folder}"
    
    # 1. Zapewnienie istnienia tabeli Delta
    spark.sql(f"CREATE TABLE IF NOT EXISTS {target_table} USING DELTA")
    
    # 2. Sprawdzenie, czy folder istnieje i czy zawiera jakiekolwiek pliki JSON
    try:
        raw_files = [f for f in os.listdir(volume_path) if f.endswith(".json")]
    except FileNotFoundError:
        raw_files = []
    
    # 3. Jeśli brak plików -> bezpieczne pominięcie bez rzucania błędu
    if not raw_files:
        print(f"⚠️  Brak plików JSON w '{volume_path}'. Pomijam ładowanie do {target_table}.")
        continue

    # 4. Wykonanie COPY INTO tylko, gdy pliki istnieją
    print(f"🔄 Ładowanie danych do {target_table} z {volume_folder} ({len(raw_files)} plików)...")
    spark.sql(f"""
        COPY INTO {target_table}
        FROM (
          SELECT 
            *,
            _metadata.file_name AS _source_file,
            current_timestamp() AS _bronze_ingested_at
          FROM '{volume_path}'
        )
        FILEFORMAT = JSON
        FORMAT_OPTIONS ('multiLine' = 'true')
        COPY_OPTIONS ('mergeSchema' = 'true')
    """)
    print(f"✅ Zakończono: {target_table}")

print("🎉 Proces warstwy Bronze zakończony pomyślnie.")